# 09 SSL Method Comparison

## Concept
One place to compare all methods under the same setup.

## Why This Method Exists
Beginners need a full-picture view of tradeoffs across SSL methods.

## Algorithm Intuition
Run supervised and four SSL variants with shared data/model settings.

## Minimal Implementation

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'src'))

from utils.seed import set_seed
from data.mnist import get_mnist_ssl_twoview
from models.small_cnn import SmallCNN
from methods.supervised import run_supervised
from methods.self_training import run_self_training
from methods.fixmatch import run_fixmatch
from methods.mean_teacher import run_mean_teacher
from methods.hybrid_teacher_threshold import run_hybrid

## Experiment

In [ ]:
set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = {
    'seed': 42,
    'labeled_per_class': 50,
    'batch_size': 128,
    'epochs': 4,
    'rounds': 4,
    'tau': 0.95,
    'tau_start': 0.7,
    'lambda_u': 1.0,
    'ema_decay': 0.99,
}
print(config)
print({'device': str(DEVICE)})

loaders = get_mnist_ssl_twoview(
    'data',
    labeled_per_class=config['labeled_per_class'],
    batch_size=config['batch_size'],
    num_workers=2,
    seed=config['seed'],
)

# Supervised
sup_model = SmallCNN()
sup_opt = torch.optim.Adam(sup_model.parameters(), lr=1e-3, weight_decay=5e-4)
sup = run_supervised(sup_model, loaders.labeled, loaders.test, sup_opt, DEVICE, epochs=config['epochs'], use_progress=True)

# Self-training
st_model = SmallCNN()
st_opt = torch.optim.Adam(st_model.parameters(), lr=1e-3, weight_decay=5e-4)
st = run_self_training(
    st_model,
    loaders.labeled,
    loaders.unlabeled,
    loaders.unlabeled_eval,
    loaders.test,
    st_opt,
    DEVICE,
    rounds=config['rounds'],
    threshold=config['tau'],
    use_soft=False,
    max_unlabeled_per_round=10000,
    use_progress=True,
)

# FixMatch-style
fm_model = SmallCNN()
fm_opt = torch.optim.SGD(fm_model.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4)
fm = run_fixmatch(
    fm_model,
    loaders.labeled,
    loaders.unlabeled,
    loaders.unlabeled_eval,
    loaders.test,
    fm_opt,
    DEVICE,
    epochs=config['epochs'],
    tau=config['tau'],
    lambda_u=config['lambda_u'],
    tau_start=config['tau_start'],
    rampup_epochs=2,
    use_progress=True,
)

# Mean Teacher
mt_student = SmallCNN()
mt_teacher = SmallCNN()
mt_opt = torch.optim.SGD(mt_student.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4)
mt = run_mean_teacher(
    mt_student,
    mt_teacher,
    loaders.labeled,
    loaders.unlabeled,
    loaders.test,
    mt_opt,
    DEVICE,
    epochs=config['epochs'],
    ema_decay=config['ema_decay'],
    lambda_u=config['lambda_u'],
    unlabeled_eval=loaders.unlabeled_eval,
    pseudo_threshold=config['tau'],
    warmup_epochs=2,
    use_progress=True,
)

# Hybrid
hy_student = SmallCNN()
hy_teacher = SmallCNN()
hy_opt = torch.optim.SGD(hy_student.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4)
hy = run_hybrid(
    hy_student,
    hy_teacher,
    loaders.labeled,
    loaders.unlabeled,
    loaders.unlabeled_eval,
    loaders.test,
    hy_opt,
    DEVICE,
    epochs=config['epochs'],
    ema_decay=config['ema_decay'],
    tau=config['tau'],
    lambda_u=config['lambda_u'],
    tau_start=config['tau_start'],
    rampup_epochs=2,
    use_progress=True,
)

## Diagnostics

In [ ]:
summary = pd.DataFrame([
    {'method': 'Supervised', 'accuracy': sup.history[-1]['val_accuracy']},
    {'method': 'Self-training', 'accuracy': st.history[-1]['val_accuracy']},
    {'method': 'FixMatch-style', 'accuracy': fm.history[-1]['val_accuracy']},
    {'method': 'Mean Teacher', 'accuracy': mt.history[-1]['val_accuracy']},
    {'method': 'Hybrid', 'accuracy': hy.history[-1]['val_accuracy']},
]).sort_values('accuracy', ascending=False)
summary

In [ ]:
plt.figure(figsize=(6, 3.4))
plt.plot([r['epoch'] for r in sup.history], [r['val_accuracy'] for r in sup.history], marker='o', label='supervised')
plt.plot([r['round'] for r in st.history], [r['val_accuracy'] for r in st.history], marker='o', label='self-training')
plt.plot([r['epoch'] for r in fm.history], [r['val_accuracy'] for r in fm.history], marker='o', label='fixmatch-style')
plt.plot([r['epoch'] for r in mt.history], [r['val_accuracy'] for r in mt.history], marker='o', label='mean teacher')
plt.plot([r['epoch'] for r in hy.history], [r['val_accuracy'] for r in hy.history], marker='o', label='hybrid')
plt.title('Method comparison: validation accuracy')
plt.xlabel('step')
plt.ylabel('accuracy')
plt.ylim(0, 1)
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()

## Key Takeaways
- This notebook provides the final side-by-side method picture.
- Compare both final accuracy and curve shape (stability).

## Failure Modes
- Results are sensitive to seed, split, and hyperparameters.
- Short runs can hide long-horizon behavior differences.